# NASDAQ Top 5 – Stock Return Analysis

**Author:** Anne Arcana  
**Date:** June 2026

---

## Overview

This project analyses the five largest NASDAQ-listed companies by market capitalisation as of mid-2026:

| Ticker | Company |
|--------|---------|
| MSFT | Microsoft |
| AAPL | Apple |
| NVDA | NVIDIA |
| AMZN | Amazon |
| GOOGL | Alphabet |

**Scope:** January 2025 – June 2026  
**Tools:** Python · pandas · yfinance · matplotlib · seaborn

**Questions we answer:**
1. How did closing prices develop over time?
2. What was the daily return for each stock?
3. Which stock delivered the highest average return?
4. Which stock carried the most risk (variance & standard deviation)?
5. How strongly do the stocks move together (correlation)?

---
## 1 · Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

---
## 2 · Load Data

In [ ]:
TICKERS     = ["MSFT", "AAPL", "NVDA", "AMZN", "GOOGL"]
START_DATE  = "2025-01-01"
END_DATE    = "2026-06-01"

raw = yf.download(TICKERS, start=START_DATE, end=END_DATE)

# yfinance ≥ 0.2 returns a MultiIndex – keep only closing prices
prices = raw["Close"]
prices.head()

---
## 3 · Closing Prices Over Time

In [ ]:
fig, ax = plt.subplots()

for ticker in prices.columns:
    ax.plot(prices.index, prices[ticker], label=ticker)

ax.set_xlabel("Date")
ax.set_ylabel("Closing Price (USD)")
ax.set_title("NASDAQ Top 5 – Closing Prices")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4 · Daily Simple Rate of Return

The daily simple return is calculated as:

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

In [ ]:
daily_returns = prices.pct_change().dropna()

fig, ax = plt.subplots()

for ticker in daily_returns.columns:
    ax.plot(daily_returns.index, daily_returns[ticker], label=ticker, alpha=0.8)

ax.set_xlabel("Date")
ax.set_ylabel("Daily Return")
ax.set_title("NASDAQ Top 5 – Daily Simple Rate of Return")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5 · Daily Returns – Individual Subplots

In [ ]:
fig, axes = plt.subplots(
    nrows=len(daily_returns.columns),
    figsize=(12, 16),
    sharex=True
)

for ax, ticker in zip(axes, daily_returns.columns):
    ax.plot(daily_returns.index, daily_returns[ticker], linewidth=0.8)
    ax.axhline(0, color='grey', linewidth=0.6, linestyle='--')
    ax.set_title(ticker)
    ax.set_ylabel("Daily Return")

axes[-1].set_xlabel("Date")
fig.suptitle("NASDAQ Top 5 – Daily Returns per Stock", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

---
## 6 · Mean Daily Return

In [ ]:
mean_returns = daily_returns.mean().sort_values(ascending=False)
print(mean_returns.to_string())

In [ ]:
fig, ax = plt.subplots()

ax.bar(mean_returns.index, mean_returns.values, color='steelblue')
ax.set_xlabel("Stock")
ax.set_ylabel("Mean Daily Return")
ax.set_title("NASDAQ Top 5 – Mean Daily Simple Rate of Return")
plt.tight_layout()
plt.show()

In [ ]:
best = mean_returns.idxmax()
print(f"Highest mean daily return: {best} ({mean_returns[best]:.4%})")

**Insight:** The stock with the highest mean daily return offered the best average reward over the period — but return alone does not account for risk. The following sections examine variance and standard deviation to put returns in context.

---
## 7 · Variance

In [ ]:
---
## 10 · Summary

| Metric | Winner (best) | Loser (worst) |
|--------|--------------|---------------|
| Mean daily return | highest = best avg. performance | lowest = weakest avg. performance |
| Variance (risk) | lowest = most stable | highest = most volatile |
| Std deviation | lowest = most stable | highest = most volatile |

**Key takeaway:** A complete investment decision requires balancing return *and* risk. The risk-adjusted return (e.g. Sharpe Ratio) would be the natural next step to compare these stocks on equal footing.

---
*© 2026 Anne Arcana · Data sourced via Yahoo Finance (yfinance) · Not financial advice.*

In [ ]:
fig, ax = plt.subplots()

ax.bar(variance.index, variance.values, color='tomato')
ax.set_xlabel("Stock")
ax.set_ylabel("Variance")
ax.set_title("NASDAQ Top 5 – Variance of Daily Returns")
plt.tight_layout()
plt.show()

In [ ]:
riskiest = variance.idxmax()
print(f"Highest variance (riskiest): {riskiest} ({variance[riskiest]:.6f})")

**Insight:** A high variance means returns are spread widely around the mean — the stock is less predictable and therefore riskier.

---
## 8 · Standard Deviation

In [ ]:
std_dev = daily_returns.std().sort_values(ascending=False)
print(std_dev.to_string())

In [ ]:
fig, ax = plt.subplots()

ax.bar(std_dev.index, std_dev.values, color='darkorange')
ax.set_xlabel("Stock")
ax.set_ylabel("Standard Deviation")
ax.set_title("NASDAQ Top 5 – Standard Deviation of Daily Returns")
plt.tight_layout()
plt.show()

In [ ]:
safest = std_dev.idxmin()
most_volatile = std_dev.idxmax()
print(f"Most stable : {safest}  (σ = {std_dev[safest]:.4%})")
print(f"Most volatile: {most_volatile} (σ = {std_dev[most_volatile]:.4%})")

**Insight:** Standard deviation is the square root of variance and is expressed in the same unit as the returns, making it more intuitive. A risk-averse investor would favour the stock with the lowest σ; a risk-tolerant investor might accept higher volatility in exchange for higher expected returns.

---
## 9 · Correlation Analysis

In [ ]:
correlation = daily_returns.corr()
print(correlation.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    ax=ax
)

ax.set_title("NASDAQ Top 5 – Return Correlation Matrix")
plt.tight_layout()
plt.show()

**Interpretation guide:**

| Correlation range | Interpretation |
|---|---|
| > 0.7 | Strong positive – stocks move closely together |
| 0.3 – 0.7 | Moderate positive – some shared movement |
| -0.3 – 0.3 | Little to no correlation |
| < -0.3 | Negative – stocks tend to move in opposite directions |

All five stocks belong to the same sector and are subject to the same macro-economic forces (interest rates, tech regulation, AI sentiment), so positive correlations across the board are expected. For portfolio diversification, pairs with *lower* correlation are preferable — they provide a natural hedge against sector-wide shocks.

---
## 10 · Summary

| Metric | Winner (best) | Loser (worst) |
|--------|--------------|---------------|
| Mean daily return | highest = best avg. performance | lowest = weakest avg. performance |
| Variance (risk) | lowest = most stable | highest = most volatile |
| Std deviation | lowest = most stable | highest = most volatile |

**Key takeaway:** A complete investment decision requires balancing return *and* risk. The risk-adjusted return (e.g. Sharpe Ratio) would be the natural next step to compare these stocks on equal footing.

---
*© 2026 Anne Arcana · Data sourced via Yahoo Finance (yfinance) · Not financial advice.*